In [1]:
!pip install requests beautifulsoup4 pandas lxml tqdm

     -------------------------------------- 80.2/80.2 kB 641.7 kB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from urllib.parse import urljoin
import time

In [3]:
urls = [
    "https://sport.detik.com/sepakbola/liga-inggris",
    "https://sport.detik.com/sepakbola/liga-italia"
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/139.0.0.0 Safari/537.36"
}

for url in urls:
    response = requests.get(url, headers=headers, timeout=10)

    print("URL:", url)
    print("Status Code:", response.status_code)
    print("Panjang halaman:", len(response.text))
    print("-" * 50)

URL: https://sport.detik.com/sepakbola/liga-inggris
Status Code: 200
Panjang halaman: 272381
--------------------------------------------------
URL: https://sport.detik.com/sepakbola/liga-italia
Status Code: 200
Panjang halaman: 272014
--------------------------------------------------


In [4]:
for url in urls:
    response = requests.get(url, headers=headers, timeout=10)
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    print("URL:", url)
    print("Judul halaman:", soup.title.get_text(strip=True))
    print("-" * 50)

URL: https://sport.detik.com/sepakbola/liga-inggris
Judul halaman: Informasi Berita Seputar Sepakbola Liga Inggris Terlengkap
--------------------------------------------------
URL: https://sport.detik.com/sepakbola/liga-italia
Judul halaman: Informasi Berita Seputar Sepakbola Liga Italia Terlengkap
--------------------------------------------------


In [5]:
data_link = []

for url in urls:
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    
    links = soup.find_all("a", href=True)

    for link in links:
        href = link["href"]
        text = link.get_text(" ", strip=True)

        if (
            "sport.detik.com/sepakbola/liga-inggris/" in href
            or "sport.detik.com/sepakbola/liga-italia/" in href
        ):
            if "/d-" in href and text:
                data_link.append({
                    "judul": text,
                    "url": href
                })

    print("Selesai crawling:", url)

df_link = pd.DataFrame(data_link)

# Hilangkan URL yang sama
df_link = df_link.drop_duplicates(subset="url")

# Reset index
df_link = df_link.reset_index(drop=True)

print("Total artikel:", len(df_link))

df_link.head(10)

Selesai crawling: https://sport.detik.com/sepakbola/liga-inggris
Selesai crawling: https://sport.detik.com/sepakbola/liga-italia
Total artikel: 103


,judul,url
0,"Hasil Carabao Cup: Drama 9 Gol, Chelsea Hajar ...",https://sport.detik.com/sepakbola/liga-inggris...
1,Millwall Vs Newcastle: Elkan Baggott Main Penu...,https://sport.detik.com/sepakbola/liga-inggris...
2,"Masih Niat Main buat Chelsea Gak, Enzo Fernandez?",https://sport.detik.com/sepakbola/liga-inggris...
3,Liverpool Punya Sponsor Baru Musim depan,https://sport.detik.com/sepakbola/liga-inggris...
4,Enzo Fernandez Bahagia Banget Tetap Dilatih Ma...,https://sport.detik.com/sepakbola/liga-inggris...
5,Xabi Alonso Redam Isu Estevao Mau Cabut dari C...,https://sport.detik.com/sepakbola/liga-inggris...
6,Michael Carrick Puas dengan Performa Rashford,https://sport.detik.com/sepakbola/liga-inggris...
7,Mac Allister Sedih Tak Ditawari Kontrak Baru d...,https://sport.detik.com/sepakbola/liga-inggris...
8,"Haaland Minim Pramusim, Khawatir Performanya T...",https://sport.detik.com/sepakbola/liga-inggris...
9,Carrick Sebut MU Baik-baik saja,https://sport.detik.com/sepakbola/liga-inggris...


In [6]:
print("Jumlah link:", len(df_link))

Jumlah link: 103


In [7]:
df_link

,judul,url
0,"Hasil Carabao Cup: Drama 9 Gol, Chelsea Hajar ...",https://sport.detik.com/sepakbola/liga-inggris...
1,Millwall Vs Newcastle: Elkan Baggott Main Penu...,https://sport.detik.com/sepakbola/liga-inggris...
2,"Masih Niat Main buat Chelsea Gak, Enzo Fernandez?",https://sport.detik.com/sepakbola/liga-inggris...
3,Liverpool Punya Sponsor Baru Musim depan,https://sport.detik.com/sepakbola/liga-inggris...
4,Enzo Fernandez Bahagia Banget Tetap Dilatih Ma...,https://sport.detik.com/sepakbola/liga-inggris...
...,...,...
98,Roma 'Rasa' Argentina,https://sport.detik.com/sepakbola/liga-italia/...
99,"Nyetel di Como, Anak Eks Pelatih Timnas Indone...",https://sport.detik.com/sepakbola/liga-italia/...
100,Fiorentina Batal Pinjam Joshua Zirkzee,https://sport.detik.com/sepakbola/liga-italia/...
101,Chivu: Stones dan Jones Masih Butuh Adaptasi,https://sport.detik.com/sepakbola/liga-italia/...


In [8]:
df_artikel = df_link[
    (
        df_link["url"].str.contains("/sepakbola/liga-inggris/", na=False)
        |
        df_link["url"].str.contains("/sepakbola/liga-italia/", na=False)
    )
    &
    df_link["url"].str.contains("/d-", na=False)
].copy()

# Hilangkan URL duplikat
df_artikel = df_artikel.drop_duplicates(subset="url")

# Reset index
df_artikel = df_artikel.reset_index(drop=True)

print("Jumlah artikel Liga Inggris + Liga Italia:", len(df_artikel))

Jumlah artikel Liga Inggris + Liga Italia: 103


In [42]:
df_artikel.head(10)

,judul,url
0,Enzo Fernandez Bahagia Banget Tetap Dilatih Ma...,https://sport.detik.com/sepakbola/liga-inggris...
1,"Haaland Minim Pramusim, Khawatir Performanya T...",https://sport.detik.com/sepakbola/liga-inggris...
2,Carrick Sebut MU Baik-baik saja,https://sport.detik.com/sepakbola/liga-inggris...
3,Xabi Alonso Redam Isu Estevao Mau Cabut dari C...,https://sport.detik.com/sepakbola/liga-inggris...
4,Michael Carrick Puas dengan Performa Rashford,https://sport.detik.com/sepakbola/liga-inggris...
5,Mac Allister Sedih Tak Ditawari Kontrak Baru d...,https://sport.detik.com/sepakbola/liga-inggris...
6,Millwall Vs Newcastle: Elkan Baggott Main Penu...,https://sport.detik.com/sepakbola/liga-inggris...
7,Arsenal Makin Pede Main Cantik,https://sport.detik.com/sepakbola/liga-inggris...
8,Ederson Angkat Bicara soal Batal Gabung ke MU,https://sport.detik.com/sepakbola/liga-inggris...
9,Ada Apa dengan Garnacho?,https://sport.detik.com/sepakbola/liga-inggris...


In [9]:
df_artikel = df_artikel.head(100)

print("Jumlah artikel yang akan di-crawl:", len(df_artikel))

Jumlah artikel yang akan di-crawl: 100


In [10]:
df_artikel.tail()

,judul,url
95,Juventus Ditinggal Kenan Yildiz 2-3 Bulan,https://sport.detik.com/sepakbola/liga-italia/...
96,"Juventus Segera Datangkan Woltemade, Lepas Dav...",https://sport.detik.com/sepakbola/liga-italia/...
97,"Lecce Vs Roma: Menang 4-0, I Lupi ke Puncak Kl...",https://sport.detik.com/sepakbola/liga-italia/...
98,Roma 'Rasa' Argentina,https://sport.detik.com/sepakbola/liga-italia/...
99,"Nyetel di Como, Anak Eks Pelatih Timnas Indone...",https://sport.detik.com/sepakbola/liga-italia/...


In [11]:
hasil_scraping = []

for i, row in tqdm(df_artikel.iterrows(), total=len(df_artikel)):
    url = row["url"]
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Ambil judul
        judul = soup.find("h1")
        judul = judul.get_text(" ", strip=True) if judul else ""
        
        # Ambil isi artikel
        artikel = soup.find("div", class_="detail__body-text")
        
        if artikel:
            isi = artikel.get_text(" ", strip=True)
        else:
            isi = ""
        
        hasil_scraping.append({
            "judul": judul,
            "url": url,
            "isi": isi
        })
        
        time.sleep(1)
        
    except Exception as e:
        print(f"Gagal scraping: {url}")
        print("Error:", e)

df_scraping = pd.DataFrame(hasil_scraping)

print("Jumlah artikel berhasil:", len(df_scraping))

 54%|█████▍    | 54/100 [01:42<03:26,  4.48s/it]

Gagal scraping: https://sport.detik.com/sepakbola/liga-italia/d-8654332/milan-punya-gaya-main-yang-jelas-di-bawah-asuhan-amorim
Error: HTTPSConnectionPool(host='sport.detik.com', port=443): Read timed out.


 98%|█████████▊| 98/100 [03:37<00:14,  7.18s/it]

Gagal scraping: https://sport.detik.com/sepakbola/liga-italia/d-8642691/lecce-vs-roma-menang-4-0-i-lupi-ke-puncak-klasemen
Error: HTTPSConnectionPool(host='sport.detik.com', port=443): Read timed out. (read timeout=10)


 99%|█████████▉| 99/100 [03:48<00:08,  8.49s/it]

Gagal scraping: https://sport.detik.com/sepakbola/liga-italia/d-8642062/roma-rasa-argentina
Error: HTTPSConnectionPool(host='sport.detik.com', port=443): Read timed out.


100%|██████████| 100/100 [03:50<00:00,  2.31s/it]

Jumlah artikel berhasil: 97


In [12]:
df_scraping.head()

,judul,url,isi
0,"Hasil Carabao Cup: Drama 9 Gol, Chelsea Hajar ...",https://sport.detik.com/sepakbola/liga-inggris...,London - Chelsea melenggang ke babak 16 besar ...
1,Millwall Vs Newcastle: Elkan Baggott Main Penu...,https://sport.detik.com/sepakbola/liga-inggris...,London - Elkan Baggott tampil penuh saat Millw...
2,"Masih Niat Main buat Chelsea Gak, Enzo Fernandez?",https://sport.detik.com/sepakbola/liga-inggris...,Jakarta - Enzo Fernandez absen lagi di laga la...
3,Liverpool Punya Sponsor Baru Musim depan,https://sport.detik.com/sepakbola/liga-inggris...,Liverpool - Tidak akan ada lagi logo Bank Stan...
4,Enzo Fernandez Bahagia Banget Tetap Dilatih Ma...,https://sport.detik.com/sepakbola/liga-inggris...,Jakarta - Manchester City sudah mendatangkan E...


In [13]:
print(df_scraping.iloc[0]["isi"])

London - Chelsea melenggang ke babak 16 besar Carabao Cup 2026/2027 usai menggasak Leeds United . The Blues menang 6-3 di kandang. Bermain di Stamford Bridge, London, Kamis (10/9/2026), Chelsea turun dengan sejumlah pemain lapis kedua. Hasilnya, pasukan Xabi Alonso cukup kesulitan meladeni Leeds. Chelsea bahkan kejebolan sejak menit ke-23. Tarik Muharemovic membawa The Whites memimpin 1-0 lewat sundulannya hingga jeda. SCROLL TO CONTINUE WITH CONTENT Di babak kedua, Leeds bahkan sempat melebarkan keunggulannya. Brendan Aaronson membawa anak asuh Daniel Farke memimpin 2-0 tiga menit selepas turun minum. Tak mau tertinggal lebih jauh, Chelsea akhirnya bangkit. Pemain macam Cole Palmer, Pedro Neto, dan Morgan Rogers bisa merespons situasi dan gantian mendominasi Leeds United . ADVERTISEMENT Palmer sempat memperkecil skor jadi 1-2 lewat penaltinya di menit ke-53. Dua menit berselang, Palmer bisa menyamakan skor menjadi 2-2 lewat sepakan kaki kirinya. Di menit ke-60, Chelsea berbalik unggul

In [14]:
print("Isi kosong:", (df_scraping["isi"] == "").sum())

Isi kosong: 0


In [15]:
print("Jumlah data:", len(df_scraping))
print("Kolom:", df_scraping.columns.tolist())

df_scraping.info()

Jumlah data: 97
Kolom: ['judul', 'url', 'isi']
<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   judul   97 non-null     str  
 1   url     97 non-null     str  
 2   isi     97 non-null     str  
dtypes: str(3)
memory usage: 2.4 KB


In [16]:
print("JUDUL:")
print(df_scraping.iloc[0]["judul"])

print("\nURL:")
print(df_scraping.iloc[0]["url"])

print("\nISI ARTIKEL:")
print(df_scraping.iloc[0]["isi"])

JUDUL:
Hasil Carabao Cup: Drama 9 Gol, Chelsea Hajar Leeds United 6-3

URL:
https://sport.detik.com/sepakbola/liga-inggris/d-8656256/hasil-carabao-cup-drama-9-gol-chelsea-hajar-leeds-united-6-3

ISI ARTIKEL:
London - Chelsea melenggang ke babak 16 besar Carabao Cup 2026/2027 usai menggasak Leeds United . The Blues menang 6-3 di kandang. Bermain di Stamford Bridge, London, Kamis (10/9/2026), Chelsea turun dengan sejumlah pemain lapis kedua. Hasilnya, pasukan Xabi Alonso cukup kesulitan meladeni Leeds. Chelsea bahkan kejebolan sejak menit ke-23. Tarik Muharemovic membawa The Whites memimpin 1-0 lewat sundulannya hingga jeda. SCROLL TO CONTINUE WITH CONTENT Di babak kedua, Leeds bahkan sempat melebarkan keunggulannya. Brendan Aaronson membawa anak asuh Daniel Farke memimpin 2-0 tiga menit selepas turun minum. Tak mau tertinggal lebih jauh, Chelsea akhirnya bangkit. Pemain macam Cole Palmer, Pedro Neto, dan Morgan Rogers bisa merespons situasi dan gantian mendominasi Leeds United . ADVERTI

In [17]:
df_scraping.to_csv(
    "hasil_scraping_detik.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Data berhasil disimpan sebagai hasil_scraping_detik.csv")

Data berhasil disimpan sebagai hasil_scraping_detik.csv
